# 🔗 Módulo 4. Integración de datasets

## Objetivo

Este notebook tiene como propósito integrar los tres conjuntos de datos previamente limpiados en un único dataset de análisis.

Antes de realizar cualquier unión, se verificará la compatibilidad entre las fuentes de información, evaluando su cobertura temporal, cobertura geográfica y las llaves de integración disponibles.

Posteriormente, se realizará la agregación del dataset del Programa de Alimentación Escolar (PAE), seguida de los procesos de integración con los datasets Principal y ETC, obteniendo un único conjunto de datos listo para el análisis exploratorio y el desarrollo de modelos de inteligencia artificial.

## Flujo del notebook

Este notebook sigue el siguiente proceso:

1. Importar librerías y módulos.
2. Cargar los datasets limpios.
3. Analizar la compatibilidad entre los datasets.
4. Verificar la cobertura temporal.
5. Verificar la cobertura geográfica.
6. Validar las llaves de integración.
7. Agregar el dataset PAE al nivel municipio-año.
8. Integrar Principal + PAE.
9. Integrar el resultado con ETC.
10. Validar el dataset final.
11. Guardar el dataset integrado.

## Importación de librerías

In [54]:
import pandas as pd
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

sys.path.append(str(PROJECT_ROOT))

from src.data.loader import DataLoader
from src.preprocessing.validator import DataValidator

## Carga de los datasets procesados

Se cargan los tres datasets generados durante el proceso de limpieza. Estos conjuntos de datos constituyen la base para la etapa de integración.

In [55]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"

loader = DataLoader(DATA_DIR)

datasets = loader.load_all_processed()

principal_clean = datasets["principal"]
pae = datasets["pae"]
etc_clean = datasets["etc"]


📂 Carpeta de datos: /Users/alejandraarciniegas/Documents/educational_crisis_ai/data/processed

📄 Cargando principal_clean.csv...
✅ Carga completada
   Filas: 15,707
   Columnas: 41

📄 Cargando pae_clean.csv...
✅ Carga completada
   Filas: 121,379
   Columnas: 11

📄 Cargando etc_clean.csv...
✅ Carga completada
   Filas: 1,335
   Columnas: 37


# Análisis de compatibilidad entre datasets

Antes de realizar los procesos de integración es necesario comprobar que los datasets sean compatibles entre sí.

En esta sección se analizarán aspectos como la cobertura temporal, la cobertura geográfica y las llaves disponibles para realizar los procesos de unión.

## Cobertura temporal

Se compara el rango de años presente en cada dataset con el fin de identificar diferencias en la disponibilidad temporal de la información.

In [56]:
print("=== Dataset Principal ===")
print(sorted(principal_clean["AÑO"].unique()))

print("\n=== Dataset PAE ===")
print(sorted(pae["AÑO"].unique()))

print("\n=== Dataset ETC ===")
print(sorted(etc_clean["AÑO"].unique()))

=== Dataset Principal ===
[np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

=== Dataset PAE ===
[np.int64(2020), np.int64(2021), np.int64(2022)]

=== Dataset ETC ===
[np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [57]:
print("Principal")
display(
    principal_clean["AÑO"]
    .value_counts()
    .sort_index()
)

print("PAE")
display(
    pae["AÑO"]
    .value_counts()
    .sort_index()
)

print("ETC")
display(
    etc_clean["AÑO"]
    .value_counts()
    .sort_index()
)

Principal


AÑO
2011    1122
2012    1122
2013    1122
2014    1122
2015    1122
2016    1122
2017    1122
2018    1122
2019    1123
2020    1122
2021    1122
2022    1121
2023    1121
2024    1122
Name: count, dtype: int64

PAE


AÑO
2020     9650
2021    27501
2022    84228
Name: count, dtype: int64

ETC


AÑO
2011    94
2012    94
2013    94
2014    94
2015    95
2016    95
2017    95
2018    96
2019    96
2020    96
2021    96
2022    96
2023    97
2024    97
Name: count, dtype: int64

## Correspondencia entre la fecha y el año en el dataset PAE

El dataset PAE contiene una columna de fecha, mientras que los demás datasets utilizan el año como referencia temporal.

En esta sección se verifica que la columna **AÑO**, generada durante la etapa de limpieza, corresponda correctamente a la fecha registrada.

In [58]:
#@title Verificar cuantos registros únicos hay por año en el dataset PAE

(
    pae[
        ["FECHA", "AÑO"]
    ]
    .drop_duplicates()
    .sort_values("FECHA")
)

,FECHA,AÑO
0,2020-12-31,2020
9672,2021-06-30,2021
19038,2021-10-31,2021
27781,2021-12-31,2021
609,2022-04-30,2022


In [59]:
#@title Verificar si los registros de 2021 corresponden a cortes administrativos de la misma información de 2020, o si corresponden a información nueva. Para esto, se listarán las fechas únicas por año.
for year in sorted(pae["AÑO"].unique()):
    print(f"\n{year}")
    print(sorted(pae.loc[
        pae["AÑO"] == year,
        "FECHA"
    ].unique()))


2020
['2020-12-31']

2021
['2021-06-30', '2021-10-31', '2021-12-31']

2022
['2022-04-30']


In [60]:
(
    pae
    .groupby("FECHA")["CANTIDAD_BENEFICIARIOS_PAE"]
    .sum()
    .sort_index()
)

FECHA
2020-12-31    5692734
2021-06-30    5612471
2021-10-31    5816766
2021-12-31    5846755
2022-04-30    3705769
Name: CANTIDAD_BENEFICIARIOS_PAE, dtype: int64

In [61]:
(
    pae
    .groupby(["AÑO", "FECHA"])
    .size()
)

AÑO   FECHA     
2020  2020-12-31     9650
2021  2021-06-30     9366
      2021-10-31     8743
      2021-12-31     9392
2022  2022-04-30    84228
dtype: int64

In [62]:
(
    pae[
        pae["MUNICIPIO"] == "Medellín"
    ]
    .groupby("FECHA")["CANTIDAD_BENEFICIARIOS_PAE"]
    .sum()
)

FECHA
2020-12-31    236645
2021-06-30    234091
2021-10-31    240823
2021-12-31    240273
2022-04-30    197204
Name: CANTIDAD_BENEFICIARIOS_PAE, dtype: int64

In [63]:
#Verificar que ocurre en 2022 que empiezan a existir tantos registros
(
    pae[pae["MUNICIPIO"] == "Medellín"]
    .groupby(["AÑO","FECHA"])
    .size()
)

AÑO   FECHA     
2020  2020-12-31      21
2021  2021-06-30      22
      2021-10-31      21
      2021-12-31      21
2022  2022-04-30    1688
dtype: int64

In [64]:
pae.groupby(
    ["AÑO","FECHA","CODIGO_MUNICIPIO"]
).size().describe()

count    5521.000000
mean       21.984966
std        70.051810
min         1.000000
25%         6.000000
50%         9.000000
75%        15.000000
max      2723.000000
dtype: float64

In [65]:
(
    pae[
        pae["AÑO"] == 2022
    ]
    .duplicated(
        subset=[
            "CODIGO_MUNICIPIO",
            "ZONA",
            "JORNADA",
            "GRUPO_POBLACIONAL",
            "CANTIDAD_BENEFICIARIOS_PAE"
        ]
    )
    .sum()
)

np.int64(40353)

In [66]:
(
    pae[pae["AÑO"] == 2022]
    .groupby(
        [
            "CODIGO_MUNICIPIO",
            "ZONA",
            "JORNADA",
            "GRUPO_POBLACIONAL"
        ]
    )
    .size()
    .sort_values(ascending=False)
    .head(20)
)

CODIGO_MUNICIPIO  ZONA    JORNADA  GRUPO_POBLACIONAL
11001             Urbana  Regular  No aplica            536
44847             Rural   Regular  Indígenas            503
11001             Urbana  Regular  Afrodescendiente     452
                                   Indígenas            440
44847             Rural   Regular  No aplica            402
11001             Urbana  Regular  Negritudes           391
05001             Urbana  Regular  No aplica            386
76001             Urbana  Regular  No aplica            352
                                   Afrodescendiente     329
                                   Negritudes           316
05001             Urbana  Regular  Afrodescendiente     297
52835             Rural   Regular  No aplica            277
11001             Urbana  Única    No aplica            270
05001             Urbana  Regular  Negritudes           256
52835             Rural   Regular  Negritudes           241
44560             Rural   Regular  Indígenas   

In [67]:
pae[
    (pae["AÑO"] == 2022)
    & (pae["CODIGO_MUNICIPIO"] == "05001")
].head(30)

,FECHA,FECHA_CORTE,CODIGO_DEPARTAMENTO,DEPARTAMENTO,CODIGO_MUNICIPIO,MUNICIPIO,ZONA,JORNADA,GRUPO_POBLACIONAL,CANTIDAD_BENEFICIARIOS_PAE,AÑO
43800,2022-04-30,30/04/2022,05,Antioquia,05001,Medellín,Urbana,Regular,No aplica,5,2022
43801,2022-04-30,30/04/2022,05,Antioquia,05001,Medellín,Urbana,Regular,Indígenas,0,2022
43802,2022-04-30,30/04/2022,05,Antioquia,05001,Medellín,Urbana,Regular,Negritudes,0,2022
43803,2022-04-30,30/04/2022,05,Antioquia,05001,Medellín,Urbana,Regular,Afrodescendiente,0,2022
43804,2022-04-30,30/04/2022,05,Antioquia,05001,Medellín,Urbana,Regular,No aplica,1,2022
43805,2022-04-30,30/04/2022,05,Antioquia,05001,Medellín,Urbana,Regular,Negritudes,0,2022
43806,2022-04-30,30/04/2022,05,Antioquia,05001,Medellín,Urbana,Regular,Afrodescendiente,0,2022
43807,2022-04-30,30/04/2022,05,Antioquia,05001,Medellín,Urbana,Regular,No aplica,73,2022
43808,2022-04-30,30/04/2022,05,Antioquia,05001,Medellín,Urbana,Regular,Negritudes,4,2022
43809,2022-04-30,30/04/2022,05,Antioquia,05001,Medellín,Urbana,Regular,Afrodescendiente,2,2022


In [68]:
pae_municipio = (
    pae
    .groupby(["AÑO", "CODIGO_MUNICIPIO"], as_index=False)
    ["CANTIDAD_BENEFICIARIOS_PAE"]
    .sum()
)

pae_municipio.groupby("AÑO").size()

AÑO
2020    1121
2021    1120
2022    1120
dtype: int64

# Análisis previo a la integración de los datasets

Antes de realizar la integración de las diferentes fuentes de información, se verificó la compatibilidad temporal y estructural de los tres conjuntos de datos.

## Cobertura temporal

Se identificó el siguiente rango de años para cada dataset:

| Dataset | Años disponibles |
|----------|------------------|
| Principal | 2011 – 2024 |
| ETC | 2011 – 2024 |
| PAE | 2020 – 2022 |

El dataset del Programa de Alimentación Escolar (PAE) únicamente contiene información para los años 2020, 2021 y 2022, mientras que los otros dos datasets abarcan el período completo 2011–2024.

Por esta razón, cualquier análisis que utilice información del PAE estará restringido a dichos años, mientras que los análisis que involucren únicamente los datasets Principal y ETC podrán realizarse para toda la serie histórica.

---

## Consistencia temporal del PAE

Se verificó la columna `FECHA` para determinar la frecuencia de actualización del dataset.

Los resultados fueron:

| Año | Fechas registradas |
|------|--------------------|
|2020|31/12/2020|
|2021|30/06/2021, 31/10/2021 y 31/12/2021|
|2022|30/04/2022|

Esto muestra que el PAE **no corresponde a una medición anual uniforme**, sino que contiene distintos cortes administrativos dependiendo del año.

En particular:

- 2020 posee un único corte anual.
- 2021 contiene tres cortes diferentes.
- 2022 contiene un único corte (abril).

Por tanto, no es posible asumir que todos los registros representan el mismo momento del año.

---

## Granularidad de los registros

Se verificó la cantidad de municipios presentes por año.

Los datasets Principal y ETC contienen aproximadamente el mismo número de municipios por año (alrededor de 1120 municipios), lo que indica una cobertura territorial prácticamente completa.

En contraste, el PAE presenta múltiples registros para un mismo municipio debido a que la información se encuentra desagregada por:

- zona (urbana/rural),
- jornada escolar,
- grupo poblacional,
- fecha de corte.

Como consecuencia, un mismo municipio puede aparecer decenas o incluso cientos de veces dentro del mismo año.

---

## Implicaciones para la integración

Debido a esta diferencia de granularidad, el dataset PAE **no puede integrarse directamente** con los datasets Principal y ETC utilizando únicamente las llaves:

- Año
- Código de municipio

Realizar el merge en este estado produciría una expansión artificial del número de registros (duplicación por producto cartesiano).

Antes de la integración será necesario construir una versión agregada del PAE que consolide la información a nivel **municipio-año**, utilizando la suma de los beneficiarios del programa.

Una vez realizada dicha agregación, el PAE tendrá la misma unidad de análisis que los otros datasets y podrá integrarse correctamente.

# Agregación del dataset PAE

El dataset del Programa de Alimentación Escolar (PAE) presenta un nivel de desagregación superior al de los demás conjuntos de datos.

Cada municipio puede aparecer múltiples veces dentro de un mismo año debido a la existencia de diferentes:

- zonas (urbana y rural),
- jornadas escolares,
- grupos poblacionales,
- fechas de corte.

Para poder integrar este dataset con los conjuntos Principal y ETC, es necesario construir una versión agregada cuya unidad de análisis sea **municipio-año**.

En esta etapa se sumará el número de beneficiarios para todas las observaciones que pertenezcan al mismo municipio y año, conservando además la información administrativa del municipio.

In [69]:
keys = ["CODIGO_MUNICIPIO", "AÑO"]

print("Registros:", len(pae))

print(
    "Combinaciones municipio-año:",
    pae[keys].drop_duplicates().shape[0]
)

print(
    "Duplicados:",
    pae.duplicated(subset=keys).sum()
)

Registros: 121379
Combinaciones municipio-año: 3361
Duplicados: 118018


In [70]:
(
    pae.groupby("CODIGO_MUNICIPIO")["MUNICIPIO"]
       .value_counts()
       .sort_values(ascending=False)
       .head(30)
)

CODIGO_MUNICIPIO  MUNICIPIO                                              
11001             Bogotá D.C.                                                2723
76001             Santiago de Cali                                           1890
05001             Medellín                                                   1773
44847             Uribia                                                     1036
52835             Tumaco                                                      910
13001             Cartagena de Indias  Distrito Turístico y Cultural          908
08001             Barranquilla  Distrito Especial  Industrial y Portuario     891
44001             Riohacha                                                    771
76109             Buenaventura                                                735
23001             Montería                                                    647
20001             Valledupar                                                  640
66001             Pereir

In [71]:
municipios = (
    pae[["CODIGO_MUNICIPIO","MUNICIPIO"]]
    .drop_duplicates()
    .sort_values("CODIGO_MUNICIPIO")
)

municipios.head(40)

,CODIGO_MUNICIPIO,MUNICIPIO
0,05001,Medellín
21,05001,Abejorral
22,05002,Abejorral
25,05002,Abriaquí
26,05004,Abriaquí
29,05004,Alejandría
30,05021,Alejandría
35,05021,Amagá
36,05030,Amagá
41,05030,Amalfi



# Tratamiento de las variables MUNICIPIO y DEPARTAMENTO

Durante el análisis exploratorio se identificó que las columnas descriptivas (MUNICIPIO y DEPARTAMENTO) y el campo CODIGO_DEPARTAMENTO presentan inconsistencias respecto al código DANE del municipio. Dado que el CODIGO_MUNICIPIO es el identificador oficial y contiene implícitamente el código del departamento (sus dos primeros dígitos), se decidió conservar únicamente esta variable como llave geográfica para la agregación e integración con los demás conjuntos de datos.


In [72]:
# =============================================================================
# Eliminación de variables descriptivas
#
# Se identificó que las columnas MUNICIPIO y DEPARTAMENTO presentan
# inconsistencias en sus nombres, mientras que los códigos DANE asociados son
# los identificadores oficiales utilizados para integrar la información con los
# demás conjuntos de datos.
#
# Por esta razón se conservan únicamente los códigos y se eliminan las
# variables de texto para evitar inconsistencias durante el análisis.
# =============================================================================

pae = pae.drop(columns=["MUNICIPIO", "DEPARTAMENTO", 'CODIGO_DEPARTAMENTO'])

print("Columnas restantes:")
display(pae.columns)

Columnas restantes:


Index(['FECHA', 'FECHA_CORTE', 'CODIGO_MUNICIPIO', 'ZONA', 'JORNADA',
       'GRUPO_POBLACIONAL', 'CANTIDAD_BENEFICIARIOS_PAE', 'AÑO'],
      dtype='object')

In [74]:
# =============================================================================
# Agregación de beneficiarios del PAE
#
# El dataset original contiene múltiples registros por municipio debido a la
# desagregación por zona, jornada y grupo poblacional. Para integrar esta
# información con los demás conjuntos de datos se calcula el número total de
# beneficiarios del PAE por municipio y año.
# =============================================================================

pae_agregado = (
    pae
    .groupby(
        ["AÑO", "CODIGO_MUNICIPIO"],
        as_index=False
    )["CANTIDAD_BENEFICIARIOS_PAE"]
    .sum()
    .rename(columns={
        "CANTIDAD_BENEFICIARIOS_PAE":"BENEFICIARIOS_PAE"
    })
)

print(f"Filas originales: {len(pae):,}")
print(f"Filas agregadas : {len(pae_agregado):,}")

display(pae_agregado.head())

# Verificación: debe existir una única fila por municipio y año
duplicados = pae_agregado.duplicated(
    subset=["AÑO", "CODIGO_MUNICIPIO"]
).sum()

print(f"\nDuplicados por municipio y año: {duplicados}")

Filas originales: 121,379
Filas agregadas : 3,361


,AÑO,CODIGO_MUNICIPIO,BENEFICIARIOS_PAE
0,2020,05001,236648
1,2020,05002,2718
2,2020,05004,349
3,2020,05021,571
4,2020,05030,1692



Duplicados por municipio y año: 0
